# 01. 시세 → 일드커브 (QuantLib으로 직접)

quanttool 없이 QuantLib만으로 미 국채 커브를 만든다. `src/quanttool/data/marketdata/yieldcurve.py`의 `YieldCurveBuilder.build`가 내부에서 하는 일을 손으로 따라가는 것이다.

| 단계 | QuantTool 코드 | QuantLib·ORE 원본 |
|---|---|---|
| 시세 한 줄 | `data/marketdata/marketdatum.py` `BondYieldQuote` | `OREData/ored/marketdata/marketdatum.hpp` |
| bill 수익률 → 가격 | `qle/termstructures.py` `bill_price` | (QuantExt 역할) |
| 헬퍼 | `YieldCurveBuilder._bill`, `_bond` | `references/QuantLib-SWIG/SWIG/ratehelpers.i` |
| 부트스트랩 | `YieldCurveBuilder.build` | `references/QuantLib-SWIG/SWIG/piecewiseyieldcurve.i` |
| 커브 빌더 설계 | 같은 곳 | `OREData/ored/marketdata/yieldcurve.cpp` |

> 데이터: `tests/fixtures/market_ust_sample.txt`는 ORE `market.txt` 형식(`날짜 키 값`)의 **합성 샘플**이다. 흐름을 익히는 용도이고, 숫자로 시장을 해석하지 않는다.

In [ ]:
from pathlib import Path

# 노트북이 어디서 열리든 레포 루트(pyproject.toml이 있는 곳)를 찾는다
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
SAMPLE = ROOT / "tests" / "fixtures" / "market_ust_sample.txt"  # 합성 샘플 (실제 시세 아님)

print(SAMPLE.read_text()[:600])

## 1. 하루치 시세 고르기
키가 스스로를 설명한다: `BOND/YIELD/USD/UST/PAR/10Y` = 채권/수익률/달러/발행자 UST/par yield/만기 10Y. `CSVLoader`가 줄마다 키를 파싱해 `BondYieldQuote`(ccy, issuer, tenor …)를 만든다.

In [ ]:
from datetime import date

import pandas as pd
import QuantLib as ql

from quanttool.data.marketdata import CSVLoader

loader = CSVLoader(SAMPLE)
ASOF = date(2026, 10, 23)
quotes = loader.load_quotes(ASOF)
q = quotes[0]
print(type(q).__name__, q.name, q.value, q.ccy, q.issuer, q.tenor)
par = {q.tenor: q.value for q in quotes if q.tenor != "6W"}
par

## 2. 관례
같은 4%라도 결제일·쿠폰 주기·일수 계산이 다르면 다른 숫자다. QuantTool은 이것을 `configs/conventions/rates.yaml`의 `UST` 항목에 둔다 (ORE `conventions.xml`).

In [ ]:
today = ql.Date(23, 10, 2026)
ql.Settings.instance().evaluationDate = today
cal = ql.UnitedStates(ql.UnitedStates.GovernmentBond)
settle = cal.advance(today, 1, ql.Days)  # T+1
bond_dc = ql.ActualActual(ql.ActualActual.Bond)
settle

## 3. bill (1Y 이하): 수익률 → 할인채 가격
재무부 bill 수익률은 bond-equivalent yield(Act/365)다. 반년 이하는 단리, 넘으면 반기복리 한 번 + 남은 기간 단리.

In [ ]:
from quanttool.qle.termstructures import bill_price  # QuantLib에 없어 qle(QuantExt 역할)에 둔 함수

helpers, labels = [], []
for t in ["1M", "2M", "3M", "4M", "6M", "1Y"]:
    mat = cal.adjust(settle + ql.Period(t), ql.Following)
    zcb = ql.ZeroCouponBond(1, cal, 100.0, mat, ql.Following, 100.0, settle)
    price = ql.QuoteHandle(ql.SimpleQuote(bill_price(par[t], mat - settle)))
    helpers.append(ql.BondHelper(price, zcb))
    labels.append(t)
len(helpers)

## 4. note·bond (2Y~30Y): par yield = 쿠폰, 가격 100
par yield의 정의가 "가격이 100이 되는 쿠폰"이므로, 쿠폰이 par yield이고 가격이 100인 가상 채권을 헬퍼로 쓴다.

In [ ]:
for t in ["2Y", "3Y", "5Y", "7Y", "10Y", "20Y", "30Y"]:
    mat = settle + ql.Period(t)
    sch = ql.Schedule(
        settle,
        mat,
        ql.Period(ql.Semiannual),
        cal,
        ql.Unadjusted,
        ql.Unadjusted,
        ql.DateGeneration.Backward,
        False,
    )
    helpers.append(
        ql.FixedRateBondHelper(
            ql.QuoteHandle(ql.SimpleQuote(100.0)),
            1,
            100.0,
            sch,
            [par[t]],
            bond_dc,
            ql.Unadjusted,
            100.0,
            settle,
        )
    )
    labels.append(t)
len(helpers)

## 5. 부트스트랩
헬퍼 13개를 동시에 맞추는 할인 함수를 만든다. 기준일은 오늘(asof)로 고정한다 — ORE `TodaysMarket`과 같다.

In [ ]:
curve = ql.PiecewiseLogCubicDiscount(today, helpers, ql.Actual365Fixed())
curve.enableExtrapolation()

d = ql.Date(17, 3, 2031)  # 시세표에 없는 아무 날짜
print(
    "DF", curve.discount(d), " zero", curve.zeroRate(d, ql.Actual365Fixed(), ql.Continuous).rate()
)

pd.DataFrame(
    {
        "tenor": labels,
        "zero_cc_%": [
            curve.zeroRate(h.maturityDate(), ql.Actual365Fixed(), ql.Continuous).rate() * 100
            for h in helpers
        ],
        "repriced_minus_quote": [h.impliedQuote() - h.quote().value() for h in helpers],
    }
).set_index("tenor")

## 6. QuantTool로 같은 일 하기
위 1~5단계가 `TodaysMarket` 하나다. 두 커브의 할인계수가 같은지 본다.

In [ ]:
from quanttool.data.configuration import load_config
from quanttool.data.marketdata import TodaysMarket

cfg = load_config(ROOT / "configs")  # 관례도 전역 등록부에 올라간다 (ORE InputParameters)
market = TodaysMarket.from_config(ASOF, cfg, loader)
ours = market.yield_curve("UST")
max(
    abs(
        ours.discount(today + ql.Period(m, ql.Months))
        - curve.discount(today + ql.Period(m, ql.Months))
    )
    for m in range(1, 361)
)

### 시세 연결: 얼린 시장 vs 연결 유지
커브 헬퍼는 loader가 들고 있는 시세 Quote(`MarketDatum.quote`)에 연결된다. `TodaysMarket`은 기본으로 구축 직후 커브를 얼린다(ORE `preserveQuoteLinkage=false`). 시세를 바꿔도 커브는 그대로다. `preserve_quote_linkage=True`면 연결이 살아 있어 시세 값만 바꿔도 커브가 따라온다. 흔들기는 보통 시뮬레이션 시장의 몫이다(04번).

In [ ]:
for linkage in [False, True]:
    ld = CSVLoader(SAMPLE)
    m = TodaysMarket.from_config(ASOF, cfg, ld, preserve_quote_linkage=linkage)
    with m.active():
        c, q = m.yield_curve("UST"), ld.get("BOND/YIELD/USD/UST/PAR/10Y", ASOF).quote
        before = c.zeroRate(10.0, ql.Continuous).rate()
        q.setValue(q.value() + 0.0010)
        moved = (c.zeroRate(10.0, ql.Continuous).rate() - before) * 1e4
    print(f"preserve_quote_linkage={linkage}: 10Y 시세 +10bp → 10Y 제로 {moved:.2f} bp")

다음: `02_todaysmarket_step_by_step.ipynb` — `TodaysMarket` 구축을 한 줄씩 연다.